# Milestone 4: Formulating MCQ Task & Fine-Tuning

**Deadline:**  `Jul 16, 2026`

- Data formatting for MCQ: Concatenating the question with options.
- Introduction to LoRA-finetuning, and its advantages over Full-finetuning.
- Setting up a training loop to fine-tune the model weights on the dataset.
- Managing GPU memory and batch sizes.
- Training efficiency strategies with Training arguments.

<!--
gdown --folder 1QO1vigWXqH_FyHiOrXU94ogUlQivRzqj
 -->

In [1]:
!uv pip install --upgrade torchao

Using Python 3.12.13 environment at: /usr
Resolved 1 package in 237ms
Prepared 1 package in 189ms
Uninstalled 1 package in 45ms
Installed 1 package in 19ms
 - torchao==0.10.0
 + torchao==0.17.0


## Import libraries

In [2]:
from functools import reduce
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForMultipleChoice,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


## Config

In [3]:
DATA_DIR = Path("dataset/")

## Helper Functions

In [4]:
def path_to_str(path: Path) -> str:
    return str(path.absolute())

## Multiple-Choice Data Formatting

In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

In [5]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": path_to_str(DATA_DIR / "train.csv"),
        "test": path_to_str(DATA_DIR / "test.csv")
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [6]:
train = dataset["train"]
test = dataset["test"]
assert isinstance(train, Dataset)

### Question 1: Label Encoding

Convert the answer column in `train.csv` into numeric labels using the following mapping:

```
A = 0
B = 1
C = 2
D = 3
E = 4
```

What is the encoded numeric label for the row at index 150?

In [7]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

In [8]:
train = train.map(
    lambda x: {"label": label_map[x["answer"]]}
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
train[150]["label"]

2

### Question 2: Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:
`str(prompt) + " [SEP] " + str(option_B)`

What is the exact character length of this formatted input string?

In [10]:
row_0 = train[0]
assert isinstance(row_0, dict)

len(row_0["prompt"] + " [SEP] " + row_0["B"])

407

## Tokenization for Multiple-Choice Models

Multiple-choice models expect inputs in the shape:
`batch_size x num_choices x sequence_length`

Since each question has five options, every row becomes five tokenized sequences.

### Question 3: Single-Row MCQ Tokenization

Using `bert-base-uncased`, tokenize the five formatted inputs for row index **0** with:

```
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"
```

After reshaping for a multiple-choice model, the final `input_ids` tensor has shape: `[1, 5, 128]`

What is the value of the second dimension?

In [11]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
assert tokenizer is not None

tokens_0 = tokenizer(
    [row_0["prompt"] + " [SEP] " + row_0[op] for op in ["A", "B", "C", "D", "E"]],
    padding = "max_length",
    truncation = True,
    max_length = 128,
    return_tensors = "pt"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
tokens_0["input_ids"].shape

torch.Size([5, 128])

In [13]:
tokens_0 = {k: v.unsqueeze(0) for k, v in tokens_0.items()}

In [14]:
tokens_0["input_ids"].shape[1]

5

### Question 4: Batch MCQ Tokenization

Tokenize the first 16 rows of `train.csv` as multiple-choice examples. \
Each row has 5 choices. \
Each choice is tokenized to length `128`.

The final input_ids tensor has shape:
`[16, 5, 128]`

How many total token positions are in this tensor?

In [15]:
first_16 = train[:16]
assert tokenizer is not None
tokens = tokenizer(
    [first_16["prompt"][r] + " [SEP] " + first_16[op][r] for r in range(16) for op in ["A", "B", "C", "D", "E"]], # noqa: E501
    padding = "max_length",
    truncation = True,
    max_length = 128,
    return_tensors = "pt"
)

In [16]:
tokens["input_ids"].shape

torch.Size([80, 128])

In [17]:
input_ids = tokens["input_ids"].view(16, 5, 128)
input_ids.shape

torch.Size([16, 5, 128])

In [18]:
reduce(lambda a, b: a*b, input_ids.shape)

10240

## Multiple-Choice Model Outputs

`AutoModelForMultipleChoice` produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to **A, B, C, D, and E**.

### Question 5: Multiple-Choice Logits

Load `bert-base-uncased` using `AutoModelForMultipleChoice`.
Tokenize row **index 0** as 5 choices and pass it through the model.

The output logits tensor has shape: `[1, 5]`

How many logits are produced for one question?

In [19]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
with torch.no_grad():
    out = model(**tokens_0)

out.logits.shape

torch.Size([1, 5])

### Question 6: Supervised Loss Tensor

For row **index 0**, pass the tokenized 5-choice input into `AutoModelForMultipleChoice` along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [21]:
with torch.no_grad():
    out = model(**tokens_0, labels=torch.tensor([train[0]["label"]]))

out.loss.shape

torch.Size([])

## LoRA for Efficient Fine-Tuning

LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

### Question 7: LoRA Trainable Parameters

Apply LoRA to the `bert-base-uncased` multiple-choice model using:

```
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS
```

Count trainable parameters using: `sum(p.numel() for p in model.parameters() if p.requires_grad)`

How many parameters are trainable?

In [22]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

295681

## Preparing Data for Hugging Face Trainer

Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized `input_ids`, `attention_mask`, and numeric `labels`.

### Question 8: Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the **first 100 rows** of `train.csv`.

For each row, create:

- `input_ids` with shape [5, 128]
- `attention_mask` with shape [5, 128]
- `labels` as the encoded answer label

For the first dataset item, `input_ids` has shape: `[5, 128]`

How many tokenized choices are stored in `input_ids`?

In [24]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
def tokenize(tokenizer, row, max_length = 128):
    tokens = tokenizer(
        [row["prompt"] + " [SEP] " + row[op] for op in ["A", "B", "C", "D", "E"]],
        padding = "max_length",
        truncation = True,
        max_length = max_length,
        return_tensors = "pt"
    )

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "labels": label_map[row["answer"]]
    }

In [26]:
train_100 = train[:100]
assert isinstance(train_100, dict)
finetune_ds = Dataset.from_dict(train_100)
finetune_ds = finetune_ds.map(
    lambda r: tokenize(tokenizer, r),
    remove_columns=finetune_ds.column_names
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [27]:
torch.Tensor(finetune_ds[0]["input_ids"]).shape

torch.Size([5, 128])

## Tiny Fine-Tuning and Inference

In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

### Question 9: Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:

```
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4
```

What is the final `global_step` reported by the Trainer?

In [28]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [29]:
train_32 = train[:32]
assert isinstance(train_32, dict)
finetune_ds = Dataset.from_dict(train_32)
finetune_ds = finetune_ds.map(
    lambda r: tokenize(tokenizer, r, 64),
    remove_columns=finetune_ds.column_names
)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [30]:
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
)

In [31]:
assert isinstance(finetune_ds, Dataset)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=finetune_ds
)

In [32]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4, training_loss=1.5924503803253174, metrics={'train_runtime': 10.697, 'train_samples_per_second': 1.496, 'train_steps_per_second': 0.374, 'total_flos': 2631086929920.0, 'train_loss': 1.5924503803253174, 'epoch': 0.5})

### Question 10: Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row **index 0** and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [33]:
device = next(model.parameters()).device
assert tokenizer is not None

tokens_0 = tokenizer(
    [row_0["prompt"] + " [SEP] " + row_0[op] for op in ["A", "B", "C", "D", "E"]],
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

tokens_0 = {
    k: v.unsqueeze(0).to(device)
    for k, v in tokens_0.items()
}

In [34]:
model.eval()

with torch.no_grad():
    out = model(**tokens_0)

round(F.softmax(out.logits, dim=-1)[0][4].item(), 4)

0.2055